In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, TimeDistributed, Conv2D
from sklearn.model_selection import train_test_split

In [ ]:
# -------------------------------------------------------------------
# Parámetros generales e Hiperparámetros del Modelo
# -------------------------------------------------------------------
input_seq_len = 5    # Número de frames de entrada (por ejemplo, 5 días)
pred_seq_len = 3     # Número de frames a predecir (por ejemplo, 3 días a futuro)
height = 64          # Altura en píxeles (ajústalo según tus datos)
width = 64           # Ancho en píxeles (ajústalo según tus datos)
channels = 3         # Número de variables (por ejemplo: AFAI, temperatura, velocidad del viento)

filters_conv1 = 32   # Número de filtros en la primera capa ConvLSTM2D
filters_conv2 = 64   # Número de filtros en la segunda capa ConvLSTM2D
filters_conv3 = 64   # Número de filtros en la tercera capa ConvLSTM2D
kernel_size = (3, 3) # Tamaño del kernel para todas las capas convolucionales
activation_fn = 'tanh'
final_activation = 'sigmoid'
learning_rate = 0.001
epochs = 10
batch_size = 4
validation_split = 0.1

In [ ]:
data_path = "sentinel_data.nc"
ds = xr.open_dataset(data_path)
AFAI = ds['AFAI'].values
temperature = ds['temperature'].values
wind_speed = ds['wind_speed'].values
def normalize(arr):
    return (arr - np.nanmin(arr)) / (np.nanmax(arr) - np.nanmin(arr))

AFAI_norm = normalize(AFAI)
temperature_norm = normalize(temperature)
wind_speed_norm = normalize(wind_speed)
data_combined = np.stack([AFAI_norm, temperature_norm, wind_speed_norm], axis=-1)
data_combined = np.nan_to_num(data_combined)

In [ ]:

def create_sequences(data, input_length, pred_length):
    X, y = [], []
    total_steps = data.shape[0]
    for i in range(total_steps - input_length - pred_length + 1):
        X.append(data[i : i + input_length])
        y.append(data[i + input_length : i + input_length + pred_length])
    return np.array(X), np.array(y)

X, y = create_sequences(data_combined, input_seq_len, pred_seq_len)
print("Forma de X:", X.shape, "Forma de y:", y.shape)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
model = Sequential()
model.add(ConvLSTM2D(filters=filters_conv1,
                     kernel_size=kernel_size,
                     input_shape=(input_seq_len, height, width, channels),
                     padding='same',
                     return_sequences=True,
                     activation=activation_fn))
model.add(BatchNormalization())
model.add(ConvLSTM2D(filters=filters_conv2,
                     kernel_size=kernel_size,
                     padding='same',
                     return_sequences=True,
                     activation=activation_fn))
model.add(BatchNormalization())
model.add(ConvLSTM2D(filters=filters_conv3,
                     kernel_size=kernel_size,
                     padding='same',
                     return_sequences=True,
                     activation=activation_fn))
model.add(BatchNormalization())
model.add(TimeDistributed(Conv2D(filters=channels,
                                 kernel_size=kernel_size,
                                 activation=final_activation,
                                 padding='same')))

optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss='mean_squared_error')
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=validation_split)
predictions = model.predict(X_test)

example_idx = 0
fig, axes = plt.subplots(2, pred_seq_len, figsize=(15, 6))
for t in range(pred_seq_len):
    axes[0, t].imshow(y_test[example_idx, t, :, :, 0], cmap='viridis')
    axes[0, t].set_title(f"Real - Día {t+1}")
    axes[0, t].axis('off')
    axes[1, t].imshow(predictions[example_idx, t, :, :, 0], cmap='viridis')
    axes[1, t].set_title(f"Predicción - Día {t+1}")
    axes[1, t].axis('off')

fig.suptitle("Comparación de Imágenes Satelitales: Real vs. Predicha (Variable AFAI)")
plt.tight_layout()
plt.show()
